# 02 · Checar qualidade
Última tarefa do job (nome: `checar_qualidade`). Compara quantas linhas do lote entraram na **bronze** e quantas sobreviveram na **silver**.

Se o descarte passar do limite (parâmetro do job `limite_descarte`, padrão **10%**), a tarefa **falha de propósito**: é melhor parar o job do que publicar número errado.

In [ ]:
from pyspark.sql.functions import col

TAB = "workspace.aula_jobs"

# Parâmetro do job (vira widget no notebook). Padrão: 10%
dbutils.widgets.text("limite_descarte", "10")
limite = float(dbutils.widgets.get("limite_descarte"))

# Valor publicado pela tarefa receber_lote
nome_lote = dbutils.jobs.taskValues.get(
    taskKey="receber_lote", key="nome_lote", default="", debugValue="")

if not nome_lote:  # rodando fora do job: avalia o lote mais recente da bronze
    nome_lote = spark.sql(f"SELECT max(arquivo_origem) FROM {TAB}.bronze_pedidos").first()[0]

print(f"Lote avaliado: {nome_lote} | limite de descarte: {limite}%")

In [ ]:
qtd_bronze = spark.table(f"{TAB}.bronze_pedidos").filter(col("arquivo_origem") == nome_lote).count()
qtd_silver = spark.table(f"{TAB}.silver_pedidos").filter(col("arquivo_origem") == nome_lote).count()

if qtd_bronze == 0:
    raise ValueError(f"O lote {nome_lote} não está na bronze. O pipeline rodou?")

descartadas = qtd_bronze - qtd_silver
pct = round(100 * descartadas / qtd_bronze, 1)

try:
    dbutils.jobs.taskValues.set(key="pct_descartado", value=pct)
except Exception:
    pass

print(f"bronze: {qtd_bronze} | silver: {qtd_silver} | descartadas: {descartadas} ({pct}%)")

In [ ]:
if pct > limite:
    raise Exception(
        f"❌ Qualidade reprovada: {pct}% das linhas de {nome_lote} foram descartadas "
        f"(limite: {limite}%). Investigue as expectations do pipeline antes de publicar."
    )

print("✅ Qualidade aprovada. Vendas acumuladas por município:")
display(spark.table(f"{TAB}.gold_vendas_municipio").orderBy(col("receita").desc()))